# NB07: Ensemble Model Training — **Exploratory / Not Part of Submitted Pipeline**

This notebook trains a meta-ensemble combining per-stressor CatBoost classifiers with organism-level genome features (proteome size, mean protein length, taxonomic order). It is **exploratory** and is **not executed or required** for the main analysis:

- Core findings rely on NB05 (per-stressor models) and NB09 (regression predictions), both of which are fully executed.
- NB07 requires `data/train_test_indices.json` (produced by NB05's global split, not currently saved) and optionally Spark access for organism-order lookup.
- Running NB07 would add ensemble predictions but is not expected to substantially improve on the per-stressor regression approach.

**To run**: ensure NB05 saves `train_test_indices.json` first, then execute this notebook in a Spark-enabled environment.

In [ ]:
import sys, json, logging, pickle, time
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models' / 'ensemble'
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
ENSEMBLE_CONFIG = {
    'SEED':42, 'TEST_SIZE':0.2, 'CALIBRATION_SIZE':0.1,
    'CATBOOST_ITERATIONS':500, 'CATBOOST_LEARNING_RATE':0.05, 'CATBOOST_DEPTH':6,
    'MIN_POSITIVES':30
}

In [ ]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

In [ ]:
# Build numeric feature matrix (same as training)
X_list = [pd.read_parquet(DATA_DIR / f"features_{n}.parquet").drop(columns=['organism'], errors='ignore')
          for n in best_combo]
X_full = pd.concat(X_list, axis=1)

# Add genome features
org_stats = labeled_pd.groupby('organism').agg(
    num_proteins=('sequence','count'),
    mean_length=('sequence', lambda s: np.mean([len(x) for x in s]))).reset_index()

# Order mapping (reuse strain_df from Spark – if not available, skip)
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    strain_df = spark.table("enigma_genome_depot_enigma.browser_strain").select("full_name","order","taxon_id").distinct().toPandas()
    org_to_order = strain_df.set_index('full_name')['order'].to_dict()
except Exception:
    org_to_order = {}
order_series = labeled_pd['organism'].map(org_to_order).fillna('Unknown')

X_ens = X_full.copy()
X_ens['order'] = order_series.values
X_ens['num_proteins'] = labeled_pd['organism'].map(org_stats.set_index('organism')['num_proteins']).values
X_ens['mean_length'] = labeled_pd['organism'].map(org_stats.set_index('organism')['mean_length']).values

In [ ]:
# Use same split as training
with open(DATA_DIR / 'train_test_indices.json', 'r') as f:
    split = json.load(f)
train_idx, test_idx = split['train_idx'], split['test_idx']

groups = labeled_pd['organism']
g_tr = groups.iloc[train_idx]

In [ ]:
def train_ensemble(stressor):
    y = labeled_pd[stressor]
    if y.sum() < ENSEMBLE_CONFIG['MIN_POSITIVES']: return None
    X_tr, X_te = X_ens.iloc[train_idx], X_ens.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    gss_cal = GroupShuffleSplit(n_splits=1, test_size=ENSEMBLE_CONFIG['CALIBRATION_SIZE'],
                                random_state=ENSEMBLE_CONFIG['SEED'])
    sub_idx, cal_idx = next(gss_cal.split(X_tr, y_tr, groups=g_tr))
    X_sub, X_cal = X_tr.iloc[sub_idx], X_tr.iloc[cal_idx]
    y_sub, y_cal = y_tr.iloc[sub_idx], y_tr.iloc[cal_idx]

    model = CatBoostClassifier(iterations=ENSEMBLE_CONFIG['CATBOOST_ITERATIONS'],
                               learning_rate=ENSEMBLE_CONFIG['CATBOOST_LEARNING_RATE'],
                               depth=ENSEMBLE_CONFIG['CATBOOST_DEPTH'],
                               cat_features=['order'], eval_metric='F1',
                               random_seed=ENSEMBLE_CONFIG['SEED'], verbose=100)
    model.fit(X_sub, y_sub, verbose=100)
    y_proba = model.predict_proba(X_te)[:,1]
    y_pred = model.predict(X_te)
    cal_proba = model.predict_proba(X_cal)[:,1]
    platt = LogisticRegression().fit(cal_proba.reshape(-1,1), y_cal)
    y_proba_cal = platt.predict_proba(y_proba.reshape(-1,1))[:,1]

    metrics = {
        'AUC': roc_auc_score(y_te, y_proba),
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_te, y_pred),
        'Pos rate': y_te.mean()
    }
    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_ensemble.cbm"))
    with open(MODEL_DIR / f"stressor_{stressor}_ensemble_platt.pkl", 'wb') as f:
        pickle.dump(platt, f)
    pd.DataFrame({'y_test': y_te.values, 'y_pred': y_pred, 'y_proba': y_proba,
                  'y_proba_cal': y_proba_cal, 'X_test_org': groups.iloc[test_idx].values}
                ).to_parquet(MODEL_DIR / f"stressor_{stressor}_ensemble_predictions.parquet")
    return metrics

In [ ]:
all_metrics = {}
for s in active_stressors:
    log.info(f"Ensemble {s}")
    m = train_ensemble(s)
    if m: all_metrics[s] = m

pd.DataFrame(all_metrics).T.to_csv(MODEL_DIR / 'ensemble_performance.csv')
log.info("Ensemble training complete.")